# 07 — Deploy with NVIDIA NIM

NVIDIA NIM packages supported models and optimized inference software as containerized microservices with standardized APIs. NIM is a deployment product: train/customize upstream, then choose a model-specific or model-free NIM path that explicitly supports the model, adapter, checkpoint format, GPU, and NIM release.

This notebook uses placeholders and never prints or stores credentials. Commands are not executed automatically.

## Prerequisites

- A supported Linux host or GPU cluster with an NVIDIA GPU and driver.
- Docker and NVIDIA Container Toolkit (`docker run --gpus all ...`).
- An NVIDIA NGC account, an API key with required services, and acceptance of the selected image/model terms.
- A NIM image tag and model profile selected from the current support matrix.
- Sufficient local cache/storage and an approved way to provide secrets.

Do not assume Qwen3-1.7B or a custom NeMo LoRA adapter is supported by every NIM image. Verify support before pulling large artifacts.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

# Boolean checks only: never print a token or include it in a command string.
print('NGC_API_KEY configured:', bool(os.getenv('NGC_API_KEY')))
print('NIM_LLM_IMAGE configured:', bool(os.getenv('NIM_LLM_IMAGE')))
print('Project root:', PROJECT_ROOT)

## Authenticate without hardcoding a secret

Set `NGC_API_KEY` using the host's approved secret mechanism. For a temporary interactive shell, enter it without saving the value to a file. Then authenticate by passing the environment variable over standard input.

POSIX shell:

```bash
test -n "$NGC_API_KEY" || { echo 'NGC_API_KEY is not set'; exit 1; }
printf '%s' "$NGC_API_KEY" | docker login nvcr.io --username '$oauthtoken' --password-stdin
```

PowerShell:

```powershell
if (-not $env:NGC_API_KEY) { throw 'NGC_API_KEY is not set' }
$env:NGC_API_KEY | docker login nvcr.io --username '$oauthtoken' --password-stdin
```

Avoid commands that place the literal key in shell history. Rotate the key immediately if it is exposed.

## Select and pull an image

Never guess a production image tag. Choose an exact model-specific or model-free image from the current NIM support matrix and set the variable in the terminal:

```bash
export NIM_LLM_IMAGE='nvcr.io/nim/<publisher>/<model>:<version>'
docker pull "$NIM_LLM_IMAGE"
```

For a current model-free NIM, the image and backend may differ (for example, vLLM or SGLang variants). Verify model source URI syntax, tokenizer/chat-template support, adapter support, and profile requirements in that release.

## Run a model-specific NIM

The model-specific pattern below passes the key from the existing environment and mounts a reusable cache. It does not contain a literal secret:

```bash
export LOCAL_NIM_CACHE="$HOME/.cache/nim"
mkdir -p "$LOCAL_NIM_CACHE"
docker run --rm --gpus all --shm-size=16GB \
  -e NGC_API_KEY \
  -v "$LOCAL_NIM_CACHE:/opt/nim/.cache" \
  -p 127.0.0.1:8000:8000 \
  "$NIM_LLM_IMAGE"
```

Loopback binding is appropriate for a workstation smoke test. Production exposure requires authentication, TLS, rate limits, network policy, monitoring, and a secret manager.

## Model-free/custom exported checkpoint pattern

If the chosen model-free NIM explicitly supports local Qwen checkpoints and the export format, mount the verified export read-only and point `NIM_MODEL_PATH` at it. The exact container image, backend, and variables must come from that release's documentation.

```bash
export EXPORTED_MODEL_DIR="$PWD/outputs/exported_hf"
docker run --rm --gpus all --shm-size=16GB \
  -e NIM_MODEL_PATH=/models/qwen \
  -e HF_TOKEN \
  -v "$EXPORTED_MODEL_DIR:/models/qwen:ro" \
  -v "$LOCAL_NIM_CACHE:/opt/nim/.cache" \
  -p 127.0.0.1:8000:8000 \
  "$NIM_LLM_MODEL_FREE_IMAGE"
```

Omit `HF_TOKEN` when the selected workflow and local model do not require it. Never invent adapter flags: consult NIM's current LoRA documentation and support matrix.

In [ ]:
CALL_NIM = False

if CALL_NIM:
    from openai import OpenAI
    client = OpenAI(base_url='http://127.0.0.1:8000/v1', api_key='not-required-for-local-smoke-test')
    available_models = client.models.list().data
    if not available_models:
        raise RuntimeError('NIM returned no models from /v1/models.')
    model_name = available_models[0].id
    response = client.chat.completions.create(
        model=model_name,
        messages=[{'role': 'user', 'content': 'Explain LoRA in one sentence.'}],
        temperature=0,
        max_tokens=96,
    )
    print('Model:', model_name)
    print(response.choices[0].message.content)
else:
    print('NIM API call skipped. Start and health-check the selected NIM first.')

## Production readiness checklist

- Pin and scan the container digest; record NIM, model, driver, and GPU profile versions.
- Keep keys in a secret manager and grant minimum required access.
- Validate base/tuned behavior with Notebook 4's held-out prompts before load testing.
- Configure health/readiness checks, resource limits, persistent cache, logs, metrics, alerts, rollback, and upgrade testing.
- Add TLS, service authentication/authorization, network policy, rate limits, and request/response data handling controls.
- For Kubernetes, use the current NIM Helm chart or NIM Operator guidance rather than translating this single-host command verbatim.

You now have the complete learning path: environment → data → NeMo LoRA → evaluation → SGLang → TensorRT-LLM → NIM.